# 8. Mochila Knapsack recursivo + Programação dinâmica

In [4]:
# Classe para o par chave-valor --------------------------------------------------
class Node:
    def __init__(self, key, value):
        self.key = key        # chave do par
        self.value = value    # valor associado
        self.next = None      # ponteiro para o próximo nó (encadeamento)


# Hash Table com encadeamento --------------------------------------------------
class HashTableChained:
    def __init__(self, capacity=4, load_factor=0.75):
        self.capacity = capacity               # número inicial de buckets
        self.size = 0                          # quantidade atual de elementos
        self.buckets = [None] * capacity       # lista de buckets (listas encadeadas)
        self.load_factor = load_factor         # limiar de carga para rehash
        self.comparacoes = 0                   # contador de comparações para teste

    # Função hash simples baseada no hash nativo do Python -------------------------------------------
    def _hash(self, key):
        return hash(key) % self.capacity

    # Rehash automático aumentando a capacidade (dobrando) -------------------------------------------
    def _rehash(self):
        old_buckets = self.buckets            # guarda os buckets antigos
        self.capacity *= 2                     # dobra a capacidade
        self.buckets = [None] * self.capacity # cria nova tabela vazia
        self.size = 0                          # reset do tamanho
        for head in old_buckets:               # percorre cada bucket antigo
            node = head
            while node:
                self.put(node.key, node.value) # insere na nova tabela
                node = node.next

    # Inserir ou atualizar um par (key, value) --------------------------------------------------
    def put(self, key, value):
        if self.size / self.capacity >= self.load_factor:
            self._rehash()                    # rehash se fator de carga exceder

        index = self._hash(key)
        node = self.buckets[index]
        prev = None

        # percorre todos os elementos procurando a chave
        while node:
            self.comparacoes += 1             # contar comparação
            # se encontrar atualiza o valor
            if node.key == key:
                node.value = value            # atualiza valor existente
                return
            prev = node
            node = node.next

        # se não eocntrar cira um novo nó no final da árvore
        new_node = Node(key, value)           # cria novo nó
        if prev:
            prev.next = new_node              # adiciona ao final da lista encadeada
        else:
            self.buckets[index] = new_node    # bucket estava vazio
        self.size += 1

    # Recuperar valor pela chave --------------------------------------------------
    def get(self, key):
        index = self._hash(key)
        node = self.buckets[index]

        # percorre todos os elementos procurando a chave
        while node:
            self.comparacoes += 1
            # se a chave do nó atual = chave procurada
            if node.key == key:
                return node.value       # rewtorna o nó atual
            node = node.next            # vai pra o próximo nó
        return None                     # chave não encontrada

    # Remover par pela chave -----------------------------------------------------
    def delete(self, key):
        index = self._hash(key)
        node = self.buckets[index]
        prev = None

        # percorre todos os elementos procurando a chave
        while node:
            self.comparacoes += 1
            # se a chave do nó atual = chave procurada
            if node.key == key:
                # se não é primeiro nó da lista
                if prev:
                    prev.next = node.next
                else:   # se é o primeiro nó do bucket
                    self.buckets[index] = node.next # bucket para o próximo nó
                self.size -= 1
                return True                   # removeu com sucesso
            prev = node
            node = node.next
        return False                          # chave não encontrada

    # Retorna o número de elementos na tabela --------------------------------------------------
    def __len__(self):
        return self.size


# (a) RECURSIVO PURO ------------------------------------------------------------
def knapsack_rec(valores, pesos, capacidade, i=0, chamadas=None, subproblemas=None):

    # mesmo padrão usado no fibonacci (Q8)
    if chamadas is None:
        chamadas = {"total": 0}

    # guarda estados diferentes (subproblemas)
    if subproblemas is None:
        subproblemas = set()

    chamadas["total"] += 1  # conta chamada

    # cada estado é definido por (i, capacidade)
    subproblemas.add((i, capacidade))

    # CASO BASE (acabou itens ou não cabe mais nada)
    if i >= len(valores) or capacidade <= 0:
        return 0, chamadas["total"], len(subproblemas)

    # se o peso não cabe, pula (igual lógica da Q6 - não escolher)
    if pesos[i] > capacidade:
        return knapsack_rec(valores, pesos, capacidade, i + 1, chamadas, subproblemas)

    # 1: escolher o item (igual "adicionar" na Q6)
    pegar, _, _ = knapsack_rec(
        valores, pesos, capacidade - pesos[i], i + 1, chamadas, subproblemas
    )
    pegar += valores[i]

    # 2: não escolher o item (igual Q6)
    nao_pegar, _, _ = knapsack_rec(
        valores, pesos, capacidade, i + 1, chamadas, subproblemas
    )

    # retorna o melhor valor
    return max(pegar, nao_pegar), chamadas["total"], len(subproblemas)


# (b) MEMOIZATION COM HASH TABLE ------------------------------------------------------------
def knapsack_memo(valores, pesos, capacidade, i=0, memo=None, chamadas=None):

    # usa a HashTableChained da Q4 no lugar de dict
    if memo is None:
        memo = HashTableChained()

    # mesmo padrão de contador da Q8
    if chamadas is None:
        chamadas = {"total": 0}

    chamadas["total"] += 1

    # chave do estado (estado mínimo suficiente) (mesma ideia de subproblemas do recursivo)
    chave = f"{i}-{capacidade}"

    # tenta pegar da hash
    valor_guardado = memo.get(chave)
    if valor_guardado is not None:
        return valor_guardado, chamadas["total"]

    # CASO BASE
    if i >= len(valores) or capacidade <= 0:
        memo.put(chave, 0)
        return 0, chamadas["total"]

    # se não cabe, pula (igual recursivo)
    if pesos[i] > capacidade:
        valor, _ = knapsack_memo(valores, pesos, capacidade, i + 1, memo, chamadas)
        memo.put(chave, valor)
        return valor, chamadas["total"]

    # 1: pegar item
    pegar, _ = knapsack_memo(
        valores, pesos, capacidade - pesos[i], i + 1, memo, chamadas
    )
    pegar += valores[i]

    # 2: não pegar
    nao_pegar, _ = knapsack_memo(
        valores, pesos, capacidade, i + 1, memo, chamadas
    )

    melhor = max(pegar, nao_pegar)

    # salva resultado na hash (memoization)
    memo.put(chave, melhor)

    return melhor, chamadas["total"]

In [5]:
# TESTANDO (mínimo 16 itens)  ------------------------------------------------------------
valores = [10, 40, 30, 50, 35, 40, 30, 45, 25, 20, 15, 60, 55, 10, 5, 70]
pesos   = [5,  4,  6,  3,  7,  2,  5,  6,  3,  2,  1,  8,  7,  2, 1,  9]

capacidade = 20

print("RESULTADO MOCHILA 0/1")
print("-----------------------------------")

# versão recursiva pura
valor_rec, chamadas_rec, sub_rec = knapsack_rec(valores, pesos, capacidade)

print("Recursivo puro:")
print("Valor máximo:", valor_rec)
print("Chamadas:", chamadas_rec)
print("Subproblemas diferentes:", sub_rec)

# versão com memoization (hash)
valor_memo, chamadas_memo = knapsack_memo(valores, pesos, capacidade)

print("\nCom memoization (HashTable):")
print("Valor máximo:", valor_memo)
print("Chamadas:", chamadas_memo)

RESULTADO MOCHILA 0/1
-----------------------------------
Recursivo puro:
Valor máximo: 225
Chamadas: 13897
Subproblemas diferentes: 275

Com memoization (HashTable):
Valor máximo: 225
Chamadas: 449
